In [0]:
from pyspark.sql.functions import col, to_date, when
import pandas as pd

## 1. Carregar da tabela bronze_dengue (sem union, pois a Bronze já consolidou)
df_bronze = spark.read.table("bronze_dengue")

## 2. Remover duplicatas usando NDUPLIC_N (flag nativa) e chave composta
df_deduplicado = df_bronze.filter((col("NDUPLIC_N").isNull()) | (col("NDUPLIC_N") != "2")).dropDuplicates(["ID_MUNICIP", "DT_NOTIFIC", "ANO_NASC", "CS_SEXO", "DT_SIN_PRI"])

## 3. Excluir nulos em colunas críticas
df_critico = df_deduplicado.dropna(subset=["DT_NOTIFIC", "ID_MUNICIP", "SG_UF_NOT", "DT_SIN_PRI"])

## 4. Preencher nulos com '9' (Ignorado) em colunas específicas
df_silver = df_critico.fillna({
    "CLASSI_FIN": "9",
    "EVOLUCAO": "9",
    "CS_GESTANT": "9",
    "CRITERIO": "9"
})

## 5. Remove colunas de controle do sistema que não servem para análise
df_silver = df_silver.drop(
    "DT_DIGITA", 
    "MIGRADO_W", 
    "FLXRECEBI", 
    "CS_FLXRET", 
    "TP_SISTEMA", 
    "ID_AGRAVO"
)

## 6. Cast de datas para o formato correto
df_silver = df_silver.withColumn("DT_NOTIFIC", to_date(col("DT_NOTIFIC")))\
                     .withColumn("DT_SIN_PRI", to_date(col("DT_SIN_PRI")))\
                     .withColumn("DT_OBITO", to_date(col("DT_OBITO")))

## 7. Enriquecimento de Dados: Tradução de códigos para facilitar o BI
df_silver = df_silver.withColumn("CS_RACA", 
    when(col("CS_RACA") == "1", "Branca")
    .when(col("CS_RACA") == "2", "Preta")
    .when(col("CS_RACA") == "3", "Amarela")
    .when(col("CS_RACA") == "4", "Parda")
    .when(col("CS_RACA") == "5", "Indígena")
    .otherwise("Ignorado")
).withColumn("CS_GESTANT",
    when(col("CS_GESTANT") == "1", "1º Trimestre")
    .when(col("CS_GESTANT") == "2", "2º Trimestre")
    .when(col("CS_GESTANT") == "3", "3º Trimestre")
    .when(col("CS_GESTANT") == "4", "Idade gestacional ignorada")
    .when(col("CS_GESTANT") == "5", "Não")
    .when(col("CS_GESTANT") == "6", "Não se aplica")
    .when(col("CS_GESTANT") == "9", "Ignorado")
    .otherwise("Ignorado")
).withColumn("TP_NOT",
    when(col("TP_NOT") == "1", "Negativa")
    .when(col("TP_NOT") == "2", "Individual")
    .when(col("TP_NOT") == "3", "Surto")
    .when(col("TP_NOT") == "4", "Agregado")
    .otherwise("Ignorado")
).withColumn("SG_UF_NOT",
    when(col("SG_UF_NOT") == "11", "Rondônia")
    .when(col("SG_UF_NOT") == "12", "Acre")
    .when(col("SG_UF_NOT") == "13", "Amazonas")
    .when(col("SG_UF_NOT") == "14", "Roraima")
    .when(col("SG_UF_NOT") == "15", "Pará")
    .when(col("SG_UF_NOT") == "16", "Amapá")
    .when(col("SG_UF_NOT") == "17", "Tocantins")
    .when(col("SG_UF_NOT") == "21", "Maranhão")
    .when(col("SG_UF_NOT") == "22", "Piauí")
    .when(col("SG_UF_NOT") == "23", "Ceará")
    .when(col("SG_UF_NOT") == "24", "Rio Grande do Norte")
    .when(col("SG_UF_NOT") == "25", "Paraíba")
    .when(col("SG_UF_NOT") == "26", "Pernambuco")
    .when(col("SG_UF_NOT") == "27", "Alagoas")
    .when(col("SG_UF_NOT") == "28", "Sergipe")
    .when(col("SG_UF_NOT") == "29", "Bahia")
    .when(col("SG_UF_NOT") == "31", "Minas Gerais")
    .when(col("SG_UF_NOT") == "32", "Espírito Santo")
    .when(col("SG_UF_NOT") == "33", "Rio de Janeiro")
    .when(col("SG_UF_NOT") == "35", "São Paulo")
    .when(col("SG_UF_NOT") == "41", "Paraná")
    .when(col("SG_UF_NOT") == "42", "Santa Catarina")
    .when(col("SG_UF_NOT") == "43", "Rio Grande do Sul")
    .when(col("SG_UF_NOT") == "50", "Mato Grosso do Sul")
    .when(col("SG_UF_NOT") == "51", "Mato Grosso")
    .when(col("SG_UF_NOT") == "52", "Goiás")
    .when(col("SG_UF_NOT") == "53", "Distrito Federal")
    .otherwise("Ignorado")
)

## 8. Padronização de Nomenclatura (Passando tudo para minúsculo)
df_silver = df_silver.toDF(*[c.lower() for c in df_silver.columns])

## 9. Dicionário para renomear colunas chave para nomes mais amigáveis
renomear_colunas = {
    "dt_notific": "data_notificacao",
    "id_municip": "codigo_municipio",
    "dt_sin_pri": "data_primeiro_sintoma",
    "ano_nasc": "ano_nascimento",
    "classi_fin": "classificacao_final",
    "cs_sexo": "sexo",
    "cs_raca": "raca",
    "cs_gestant": "gestante"
}

for antigo, novo in renomear_colunas.items():
    df_silver = df_silver.withColumnRenamed(antigo, novo)

## 10. Salvar como projeto_dengue.default.silver_dengue_consolidado
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_dengue_consolidado")
display(df_silver)